# AGARAM — Tamil Handwritten Character Recognition Model Training

This notebook provides the complete training, evaluation, and TensorFlow Lite conversion pipeline for recognizing 31 Tamil handwritten characters (12 Uyir + 18 Mei + 1 Aytham).

**Target Characters (31 Classes):**
- **Uyir (12):** அ, ஆ, இ, ஈ, உ, ஊ, எ, ஏ, ஐ, ஒ, ஓ, ஔ
- **Mei (18):** க், ங், ச், ஞ், ட், ண், த், ந், ப், ம், ய், ர், ல், வ், ழ், ள், ற், ன்
- **Aytham (1):** ஃ

In [ ]:
# 1. Install Dependencies
!pip install -q tensorflow numpy pillow opencv-python scikit-learn matplotlib seaborn

In [ ]:
# 2. Imports & Setup
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# 3. Load 31 Class Labels
LABELS = [
    'அ', 'ஆ', 'இ', 'ஈ', 'உ', 'ஊ', 'எ', 'ஏ', 'ஐ', 'ஒ', 'ஓ', 'ஔ',
    'க்', 'ங்', 'ச்', 'ஞ்', 'ட்', 'ண்', 'த்', 'ந்', 'ப்', 'ம்',
    'ய்', 'ர்', 'ல்', 'வ்', 'ழ்', 'ள்', 'ற்', 'ன்', 'ஃ'
]
NUM_CLASSES = len(LABELS)
print(f"Total classes: {NUM_CLASSES}")

In [ ]:
# 4. Build Compact CNN Model
def build_model(input_shape=(32, 32, 1), num_classes=31):
    inputs = layers.Input(shape=input_shape, name="stroke_input")
    
    # Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.SpatialDropout2D(0.20)(x)
    
    # Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.SpatialDropout2D(0.25)(x)
    
    # Block 3
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.SpatialDropout2D(0.30)(x)
    
    # Dense Head
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="TamilCNN")

model = build_model()
model.summary()

In [ ]:
# 5. Export to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("tamil_handwriting_model.tflite", "wb") as f:
    f.write(tflite_model)

# Quantized TFLite
converter_quant = tf.lite.TFLiteConverter.from_keras_model(model)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter_quant.convert()

with open("tamil_handwriting_model_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

print(f"Float32 TFLite Size: {len(tflite_model) / 1024:.1f} KB")
print(f"Quantized TFLite Size: {len(tflite_quant_model) / 1024:.1f} KB")